# β-VAE Training

## Clone Repository & Install Dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/nanokwok/Deep-Loan-Default-VAE.git'
REPO_DIR = '/content/Deep-Loan-Default-VAE'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already cloned — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU')

## Import Data

In [ ]:
import os, json

# TODO: Replace with your Kaggle credentials before running. Do NOT commit.
KAGGLE_USERNAME = 'your_kaggle_username'
KAGGLE_KEY      = 'your_kaggle_api_key'

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials saved.')

In [ ]:
import os, subprocess, sys

RAW_DIR = 'data/raw'
RAW_CSV = f'{RAW_DIR}/accepted_2007_to_2018Q4.csv'
os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(RAW_CSV):
    size_gb = os.path.getsize(RAW_CSV) / 1e9
    print(f'Raw CSV already present ({size_gb:.2f} GB) — skipping download.')
else:
    print('Downloading Lending Club dataset from Kaggle (~1.6 GB)...')
    # Dataset: https://www.kaggle.com/datasets/wordsforthewise/lending-club
    subprocess.run([
        sys.executable, '-m', 'kaggle', 'datasets', 'download',
        'wordsforthewise/lending-club',
        '--file', 'accepted_2007_to_2018Q4.csv',
        '-p', RAW_DIR, '--unzip',
    ], check=True)
    size_gb = os.path.getsize(RAW_CSV) / 1e9
    print(f'Download complete: {size_gb:.2f} GB')

## Run Preprocessing Pipeline

Runs `src/preprocess.py` to generate the `.npy` feature arrays.

Steps: load CSV → filter to 6 continuous features → 80/20 stratified split → impute (median) → log1p → winsorize (p1/p99) → StandardScaler → save.

Expected runtime: ~3–5 min.

In [ ]:
import os, subprocess, sys, numpy as np, json
from pathlib import Path

PROC_DIR = Path('data/processed')
REQUIRED = ['X_train.npy', 'X_val.npy', 'y_train.npy', 'y_val.npy',
            'feature_columns.json', 'scaler.pkl']

if all((PROC_DIR / f).exists() for f in REQUIRED):
    print('Processed arrays already exist — skipping. Delete data/processed/ to rerun.')
else:
    print('Running preprocessing (~3-5 min)...')
    subprocess.run([sys.executable, '-m', 'src.preprocess'], check=True)

X_train = np.load(PROC_DIR / 'X_train.npy')
X_val   = np.load(PROC_DIR / 'X_val.npy')
y_val   = np.load(PROC_DIR / 'y_val.npy')
with open(PROC_DIR / 'feature_columns.json') as f:
    feat_names = json.load(f)

print(f'X_train   : {X_train.shape}  dtype={X_train.dtype}')
print(f'X_val     : {X_val.shape}  anomaly_rate={y_val.mean()*100:.1f}%')
print(f'features  : {feat_names}')
print(f'max|x|    : {abs(X_train).max():.3f}  (should be ≤ 6.0 after log1p+winsorize+scale)')
assert not np.isnan(X_train).any(), 'NaN detected in X_train!'
assert abs(X_train).max() <= 6.0,  f'Outlier check failed: max|x|={abs(X_train).max():.3f}'
print('All checks passed.')

## Verify GPU & Model Architecture

In [ ]:
import torch, numpy as np
from pathlib import Path

device = (
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)
print(f'Device   : {device}')
if device == 'cuda':
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
elif device == 'cpu':
    print('WARNING: CPU only — training will be slower.')

proc      = Path('data/processed')
X_train   = np.load(proc / 'X_train.npy')
y_val     = np.load(proc / 'y_val.npy')
input_dim = X_train.shape[1]
print(f'input_dim  : {input_dim}')
print(f'train rows : {len(X_train):,}  (normal only)')
print(f'val normal : {(y_val==0).sum():,}   anomaly : {(y_val==1).sum():,}')

import sys; sys.path.insert(0, '.')
from src.model import BetaVAE
from src.config import BETA, LATENT_DIM, ENCODER_DIMS, DECODER_DIMS
model    = BetaVAE(input_dim=input_dim)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nBetaVAE architecture:')
print(f'  Encoder : {input_dim} → {” → “.join(str(d) for d in ENCODER_DIMS)} → μ/logσ² ({LATENT_DIM})')
print(f'  Decoder : {LATENT_DIM} → {” → “.join(str(d) for d in DECODER_DIMS)} → {input_dim}')
print(f'  β={BETA}  latent_dim={LATENT_DIM}  params={n_params:,}')

## Train the VAE

- Trains on normal (Fully Paid) rows only.
- Early stopping monitors **val normal-only** reconstruction loss (patience = 10).
- Best checkpoint saved to `models/best_vae.pth` and the experiment folder.
- Uncomment the `cfg.EXPERIMENTS_DIR` line to persist results to Google Drive.

In [ ]:
import sys, importlib
if '.' not in sys.path:
    sys.path.insert(0, '.')

import src.config as cfg
# Uncomment to persist experiments across Colab VM resets:
# cfg.EXPERIMENTS_DIR = '/content/drive/MyDrive/Loan_VAE_Project'

# Reload src modules so config changes take effect
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'):
        del sys.modules[mod]

from src.train import train
exp_dir = train()   # returns Path to this run's experiment folder

print()
print('Experiment artefacts saved to:', exp_dir)
import os
for fname in sorted(os.listdir(str(exp_dir))):
    size = os.path.getsize(os.path.join(str(exp_dir), fname))
    print(f'  {fname:<30} {size/1024:.1f} KB')

## Loss Curves

In [ ]:
# Display plots saved by train() directly from the experiment folder
from IPython.display import Image, display

for fname in ['loss_curves.png', 'reconstruction_plot.png']:
    img_path = exp_dir / fname
    if img_path.exists():
        print(f'--- {fname} ---')
        display(Image(str(img_path)))
    else:
        print(f'{fname} not found.')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_csv = exp_dir / 'loss_history.csv'
log_df  = pd.read_csv(log_csv)
print(f'Epochs trained: {len(log_df)}')
print(log_df[['epoch', 'train_total', 'val_total',
              'train_recon', 'val_recon', 'train_kl', 'val_kl']].tail(5).to_string(index=False))

best_ep = int(log_df.loc[log_df['val_total'].idxmin(), 'epoch'])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, tr_col, vl_col, title in zip(
    axes,
    ['train_total', 'train_recon', 'train_kl'],
    ['val_total',   'val_recon',   'val_kl'],
    ['Total Loss (recon + β·KL)', 'Reconstruction Loss (MSE)', 'KL Divergence'],
):
    ax.plot(log_df['epoch'], log_df[tr_col], label='train',            color='steelblue')
    ax.plot(log_df['epoch'], log_df[vl_col], label='val (normal only)', color='crimson', linestyle='--')
    ax.axvline(best_ep, color='orange', linewidth=1.2, linestyle=':', label=f'best ep {best_ep}')
    ax.set_xlabel('Epoch')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle(f'β-VAE Training Curves  (best epoch={best_ep})', fontsize=13)
plt.tight_layout()
import os; os.makedirs('reports/figures', exist_ok=True)
plt.savefig('reports/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best epoch: {best_ep}  val_loss={log_df["val_total"].min():.4f}')

## Anomaly Score Preview

Reconstruction error distributions for Normal vs Anomaly rows in the val set.

**Expected:** Charged Off histogram shifted right of Fully Paid.
**If heavily overlapping:** lower `BETA` in `src/config.py` and retrain.

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from src.model import BetaVAE
from src.config import DATA_PROC_DIR

proc   = Path(DATA_PROC_DIR)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt  = torch.load('models/best_vae.pth', map_location=device)
model = BetaVAE(input_dim=ckpt['input_dim']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Checkpoint: epoch={ckpt['epoch']}  val_loss={ckpt['val_loss']:.4f}  β={ckpt['beta']}")

X_val   = torch.from_numpy(np.load(proc / 'X_val.npy').astype('float32')).to(device)
y_val   = np.load(proc / 'y_val.npy')

with torch.no_grad():
    x_hat, _, _ = model(X_val)
    errors = ((X_val - x_hat) ** 2).mean(dim=1).cpu().numpy()

normal_e  = errors[y_val == 0]
anomaly_e = errors[y_val == 1]
sep = anomaly_e.mean() / (normal_e.mean() + 1e-9)
print(f'Normal  mean={normal_e.mean():.4f}  p95={np.percentile(normal_e, 95):.4f}')
print(f'Anomaly mean={anomaly_e.mean():.4f}  p95={np.percentile(anomaly_e, 95):.4f}')
print(f'Separation ratio: {sep:.2f}×  (>1.5 is good)')

clip = float(np.percentile(errors, 99))
bins = np.linspace(0, clip, 80)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_e.clip(max=clip),  bins=bins, alpha=0.6, density=True,
        color='steelblue', label=f'Fully Paid (n={len(normal_e):,})')
ax.hist(anomaly_e.clip(max=clip), bins=bins, alpha=0.6, density=True,
        color='crimson',   label=f'Charged Off (n={len(anomaly_e):,})')
ax.set_xlabel('Reconstruction Error (mean squared per feature)')
ax.set_ylabel('Density')
ax.set_title(f'Anomaly Score Distribution — Val Set  (separation {sep:.2f}×)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('reports/figures/anomaly_score_preview.png', dpi=150, bbox_inches='tight')
plt.show()